# **PyTorch Day-6**
**In this notebook we will learn how to implement a transformer based GPT model**

**Our goal is to build a model with gpt2 structure but smaller**


In [2]:
try:
    import datasets
except ImportError:
    !pip install datasets
    import datasets

In [3]:
try:
    import transformers
except ImportError:
    %pip install transformers
    import transformers

In [448]:
import torch
from torch import nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

* # Checking for gpu
**I am running this on a mac so I use Apple Silicon GPU (MPS)**

In [5]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon GPU (MPS)")
else:
    device = torch.device("cpu")
    print("MPS not available. Using CPU.")

Using Apple Silicon GPU (MPS)


* # Data
**We use the TinyStories data set**

In [6]:
train_data = datasets.load_dataset("roneneldan/TinyStories", split="train")
test_data = datasets.load_dataset("roneneldan/TinyStories", split="validation")
print('------------------------------------------------------------')
print('Training data:\n',train_data)
print('Testing data:\n',test_data)
print('------------------------------------------------------------')
print('An example of training data:\n',train_data[0])
print('An example of testing data:\n',test_data[0])

------------------------------------------------------------
Training data:
 Dataset({
    features: ['text'],
    num_rows: 2119719
})
Testing data:
 Dataset({
    features: ['text'],
    num_rows: 21990
})
------------------------------------------------------------
An example of training data:
 {'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and w

* ### Tokenization
**For tokenization we use the gpt2 tokenizer**

In [7]:
from transformers import AutoTokenizer, TextStreamer
# Load the GPT-2 tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# GPT-2 does not have a padding token.
# We use the End of sentence (EOS) token for padding
tokenizer.pad_token = tokenizer.eos_token

# Defining a tokenizing function
def tokenizing_fn(data):
    return tokenizer(
        data["text"],
        truncation=True,
        max_length=512,
        padding= False
    )

# Tokenize the datasets
train_data_tokens = train_data.map(
    tokenizing_fn,
    batched=True,
    remove_columns=["text"],
    batch_size= 32
)

test_data_tokens = test_data.map(
    tokenizing_fn,
    batched=True,
    remove_columns=["text"],
    batch_size= 32
)

In [8]:
total_train_tokens = sum(
    len(story["input_ids"])
    for story in train_data_tokens
)


total_test_tokens = sum(
    len(story["input_ids"])
    for story in test_data_tokens
)

print("Total training tokens:", total_train_tokens)

print("Total testing tokens:", total_test_tokens)

Total training tokens: 463939980
Total testing tokens: 4675638


In [304]:
# Looking at an example of data
# from transformers import TextStreamer
# streamer = TextStreamer(tokenizer, skip_prompt=True)

story = test_data_tokens[2000]
print(story.keys())
token_ids = story["input_ids"]
tokens = tokenizer.convert_ids_to_tokens(token_ids)

for token_id, token in zip(token_ids, tokens):
    print(f"story token ids = {token_id:6d} , story tokes = {token.replace("Ġ", " ").replace("Ċ", "\n")}")

dict_keys(['input_ids', 'attention_mask'])
story token ids =  14295 , story tokes = Jack
story token ids =    373 , story tokes =  was
story token ids =    379 , story tokes =  at
story token ids =    262 , story tokes =  the
story token ids =  10481 , story tokes =  beach
story token ids =    351 , story tokes =  with
story token ids =    465 , story tokes =  his
story token ids =   1641 , story tokes =  family
story token ids =     13 , story tokes = .
story token ids =    679 , story tokes =  He
story token ids =  12408 , story tokes =  wore
story token ids =    465 , story tokes =  his
story token ids =    649 , story tokes =  new
story token ids =   4171 , story tokes =  blue
story token ids =  12581 , story tokes =  pants
story token ids =    475 , story tokes =  but
story token ids =    339 , story tokes =  he
story token ids =   2936 , story tokes =  felt
story token ids =    845 , story tokes =  very
story token ids =  12916 , story tokes =  uncomfortable
story token ids =    

* ### Preparing the data to pass to a torch model

*  **Let's concatenate the data in torch tensors**

In [15]:
batch_size = 32
context_size = 64

# total_train_tokens = total_train_tokens // 10
# total_test_tokens = total_test_tokens // 10


train_set = torch.zeros(total_train_tokens, dtype=torch.int64)
test_set = torch.zeros(total_test_tokens, dtype=torch.int64)

start_index = 0
end_index = 0
for i , story in enumerate(train_data_tokens):
    story_tensor = torch.tensor(story["input_ids"]).to(torch.int64)
    end_index = end_index + story_tensor.shape[0]  
    train_set[start_index:end_index] = story_tensor[:]
    start_index = end_index

start_index = 0
end_index = 0
for i , story in enumerate(test_data_tokens):
    story_tensor = torch.tensor(story["input_ids"])
    end_index = end_index + story_tensor.shape[0]  
    test_set[start_index:end_index] = story_tensor[:]
    start_index = end_index


n_test = (test_set.shape[0]//(batch_size * context_size)) * (batch_size * context_size)
n_train = (train_set.shape[0]//(batch_size * context_size)) * (batch_size * context_size)

train_set = train_set[:n_train]
test_set = test_set[:n_test]

* **Reshaping the training data into the format to pass to the embedding layer in the batch loop**
  
It should be a tensor of shape (N_batches, batch_size =32, context_size=64) 

In [52]:
N_batches = n_train // (batch_size * context_size)
print(f"Number of batches={N_batches}")
train_set_r = train_set.reshape(N_batches , batch_size , context_size)
train_set_r.shape
vocab_size = train_set_r.max()
print(f"Vocab size= {vocab_size}")
print(f"Shape of the training set={train_set_r.shape}")

Number of batches=226533
Vocab size= 50255
Shape of the training set=torch.Size([226533, 32, 64])


* ### Sending the data to device

In [210]:
train_set_r = train_set_r.to(device)

* # Creating the model

* ### Model configuration

In [389]:
@dataclass
class DataConfig:
    N_batches: int 
    batch_size: int
    vocab_size: int
    context_size: int

@dataclass
class ModelConfig:
    context_size: int
    embed_dim: int
    init_std: float
    eps: float
    N_heads: int
    attn_dim: int
    dropout: float
    MLP_hidden_size: float
    N_layers: int
    
    
d_cfg = DataConfig(N_batches =  N_batches,
                   batch_size = batch_size,
                   vocab_size = vocab_size.item(),
                   context_size =context_size
                  ) 

m_cfg = ModelConfig(context_size =context_size,
                    embed_dim=512,
                    init_std=0.02,
                    eps=1e-5,
                    N_heads = 4,
                    attn_dim = 128,
                    dropout = 0.1,
                    MLP_hidden_size = 1024,
                    N_layers = 4
                   )

print(m_cfg)
print(d_cfg)

ModelConfig(context_size=64, embed_dim=512, init_std=0.02, eps=1e-05, N_heads=4, attn_dim=128, dropout=0.1, MLP_hidden_size=1024, N_layers=4)
DataConfig(N_batches=226533, batch_size=32, vocab_size=50255, context_size=64)


* ### The embedding layer

In [268]:
class Embedding(nn.Module):
    def __init__(self, m_cfg:ModelConfig , d_cfg:DataConfig):
        super().__init__()
        vocab_size = d_cfg.vocab_size 
        embed_dim = m_cfg.embed_dim
        context_size = m_cfg.context_size

        # Token embedding
        self.token_embedding = nn.Parameter(
            torch.empty(vocab_size, embed_dim)
        )
        # Positional embedding
        self.position_embedding = nn.Parameter(
            torch.empty(context_size, embed_dim)
        )

        # Initialization
        nn.init.normal_(self.token_embedding, mean=0.0, std=m_cfg.init_std)
        nn.init.normal_(self.position_embedding, mean=0.0, std=m_cfg.init_std)

    def forward(self, input_ids):

        position_ids = torch.arange(context_size, device=input_ids.device)
        
        x = self.token_embedding[input_ids] + self.position_embedding[position_ids]

        return x       

**Taking a look at the embedding layer**

In [269]:
model_embed = Embedding(m_cfg , d_cfg)
model_embed.to(device)
model_embed.eval()
X=train_set_r[0,:,:].squeeze()
print(f"Input is on device:{X.device}")
model_embed.eval()
with torch.inference_mode():
    Y_embed = model_embed(X)
print(f"Shape of the embedding layer output= {Y_embed.shape}")
model_embed.state_dict()
print(f"Output is on device:{Y_embed.device}")

Input is on device:mps:0
Shape of the embedding layer output= torch.Size([32, 64, 512])
Output is on device:mps:0


* ### The normalization layer

In [270]:
class Normalize(nn.Module):
    def __init__(self, m_cfg:ModelConfig):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(m_cfg.embed_dim))
        self.beta = nn.Parameter(torch.zeros(m_cfg.embed_dim))
        self.eps = m_cfg.eps
        
    def forward(self, x: torch.Tensor):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)

        x = (x - mean) / torch.sqrt(var + self.eps)

        return self.gamma * x + self.beta

**Looking at the Normalization layer**

In [271]:
model_norm = Normalize(m_cfg)
model_norm.to(device)
model_norm.eval()
with torch.inference_mode():
    Y_norm = model_norm(Y_embed)

print(f"Normalization layer output shape is {Y_norm.shape}")
print(f"Output is on device:{Y_norm.device}")
model_norm.state_dict()


Normalization layer output shape is torch.Size([32, 64, 512])
Output is on device:mps:0


OrderedDict([('gamma',
              tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
                      1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
                      1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
                      1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
                      1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
                      1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
                      1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
                      1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
                      1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
                      1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
                      1., 1., 1., 1.,

* ### The Attention layer
**This is the most exciting part of the transformer**

In [316]:
class Attention(nn.Module):
    def __init__(self, m_cfg:ModelConfig):
        super().__init__()
        
        
        self.W_keyes = nn.Parameter(
            torch.empty((m_cfg.N_heads, m_cfg.attn_dim, m_cfg.embed_dim))
        ) #<- To map the embedded model vectors to attention keyes

        self.b_keyes = nn.Parameter(
            torch.zeros((m_cfg.N_heads, m_cfg.attn_dim))
        ) #<- bias for keyes

        self.W_queries = nn.Parameter(
            torch.empty((m_cfg.N_heads, m_cfg.attn_dim, m_cfg.embed_dim))
        ) #<- To map the embedded model vectors to attention queries

        self.b_queries = nn.Parameter(
            torch.zeros((m_cfg.N_heads, m_cfg.attn_dim))
        ) #<- bias for queries
        
        self.W_values = nn.Parameter(
            torch.empty((m_cfg.N_heads, m_cfg.attn_dim, m_cfg.embed_dim))
        ) #<- To map the embedded model vectors to attention values

        self.b_values = nn.Parameter(
            torch.zeros((m_cfg.N_heads, m_cfg.attn_dim))
        ) #<- bias for values
        
        self.W_output = nn.Parameter(
            torch.empty((m_cfg.N_heads * m_cfg.attn_dim, m_cfg.embed_dim))
        ) #<- To map from attention space back to embedding space 
        
        self.b_output = nn.Parameter(
            torch.zeros((m_cfg.embed_dim))
        ) #<- output bias

        # initializing with standard Gaussian random numbers
        nn.init.normal_(self.W_queries, std=m_cfg.init_std)
        nn.init.normal_(self.W_keyes, std=m_cfg.init_std)
        nn.init.normal_(self.W_values, std=m_cfg.init_std)
        nn.init.normal_(self.W_output, std=m_cfg.init_std)

        
        self.register_buffer("neg_inf",
                             torch.tensor(float("-inf"),
                             dtype=torch.float32)
                            ) #<- will be used later for causal masking
        
        self.register_buffer("mask",
                            torch.triu(
                                torch.ones(size=(m_cfg.context_size , m_cfg.context_size) , device=device ),
                                diagonal=1).bool()
                            ) #<- The causal mask
        
        self.m_cfg = m_cfg #<- owning the cfg info
        self.attention_dropout = nn.Dropout(m_cfg.dropout)
        
    def forward(self, x):

        # Mapping vectors from the embedding space to attention space.
        ## for the following we can also use torch.einsum() 
        x_k = torch.tensordot(x, self.W_keyes, dims=([-1], [-1])) + self.b_keyes   
        x_q = torch.tensordot(x, self.W_queries,  dims=([-1], [-1])) + self.b_queries
        x_v = torch.tensordot(x, self.W_values, dims=([-1], [-1])) + self.b_values
        # print(f"Shape of x_v is {x_v.shape}")

        
        attn_scores = torch.einsum(
            'bqha,bkha -> bhqk',
            x_q , x_k) #<- contract on the attention dimension (dot product of queries and keyes)
        
        attn_scores /= self.m_cfg.attn_dim**0.5 #<- Normalizing with square root of attention space dimension

        q = x.shape[1]
        attn_scores = attn_scores.masked_fill_(
            self.mask[:q,:q],
            self.neg_inf
        ) #<- applying the causal mask
        # print(f"Shape of attention scores is {attn_scores.shape}")
        
        attn_probs = attn_scores.softmax(dim = -1) #<- scores to probabilities so that sum of each row adds up to one
        attn_probs = self.attention_dropout(attn_probs)

        V = torch.einsum(
            'bhqv,bvha->bqha',
            attn_probs , x_v
        )#<- computing the weighted values 
        
        b, q, h, a = V.shape
        V = V.reshape(b, q, h * a) #<- Concatenating all the heads
        # print(f"Shape of wighted values is {V.shape}")
        
        output = torch.einsum(
            'ce,bvc->bve',
            self.W_output , V
        ) + self.b_output
        return output

**Looking at the Attention class**

In [317]:
model_attn = Attention(m_cfg)
model_attn.to(device)
model_attn.eval()
print(f"Shape of the Keyes matrix = {model_attn.W_keyes.shape}")
print(f"Shape of the input matrix = {Y_norm.shape}")
with torch.inference_mode():
    output = model_attn(Y_norm[:,:,:])
print(f"Attention layer output's shape is {output.shape}")


Shape of the Keyes matrix = torch.Size([4, 128, 512])
Shape of the input matrix = torch.Size([32, 64, 512])
Attention layer output's shape is torch.Size([32, 64, 512])


* ### The MLP layer
**MLP acts as a tokenwise nonlinearity that acts on individual token vectors**

**The same MLP acts on all token vectors in all input positions**

In [328]:
class MLP(nn.Module):
    def __init__(self, m_cfg):
        super().__init__()

        self.linear1 = nn.Linear(
            m_cfg.embed_dim,
            m_cfg.MLP_hidden_size
        )

        self.linear2 = nn.Linear(
            m_cfg.MLP_hidden_size,
            m_cfg.embed_dim
        )

        self.activation = nn.GELU()

    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)
        return x

**Looking at the MLP layer**

In [330]:
model_MLP = MLP(m_cfg)
model_MLP.to(device)
model_MLP.eval()
with torch.inference_mode():
    MLP_out = model_MLP(output)

print(list(model_MLP.state_dict()))
print(MLP_out.shape)

['linear1.weight', 'linear1.bias', 'linear2.weight', 'linear2.bias']
torch.Size([32, 64, 512])


* ### Putting together the transformer block
   **This will take the output of the embedding model as input.**

In [352]:
class TransformerBlock(nn.Module ):
    def __init__(self, m_cfg: ModelConfig):
        super().__init__()
        self.norm1 = Normalize(m_cfg)
        self.attn = Attention(m_cfg)
        self.norm2 = Normalize(m_cfg)
        self.mlp = MLP(m_cfg)
        self.m_cfg = m_cfg

    def forward(self , x):
        y = self.norm1(x)
        y = self.attn(y) + x #<- adding the residual after attention

        y = self.norm2(y)
        y = self.mlp(y)

        return y
        

**Looking at the transformer block**

In [354]:
model_transformer = TransformerBlock(m_cfg)
model_transformer.to(device)
print( list( model_transformer.state_dict() ) )

model_transformer.eval()
with torch.inference_mode():
    transformer_output = model_transformer(Y_embed)

print(transformer_output.shape)

['norm1.gamma', 'norm1.beta', 'attn.W_keyes', 'attn.b_keyes', 'attn.W_queries', 'attn.b_queries', 'attn.W_values', 'attn.b_values', 'attn.W_output', 'attn.b_output', 'attn.neg_inf', 'attn.mask', 'norm2.gamma', 'norm2.beta', 'mlp.linear1.weight', 'mlp.linear1.bias', 'mlp.linear2.weight', 'mlp.linear2.bias']
torch.Size([32, 64, 512])


* ### The unembedding layer

In [363]:
class Unembed(nn.Module):
    def __init__(self,m_cfg:ModelConfig , d_cfg:DataConfig):
        super().__init__()
        self.unembed = nn.Linear(in_features = m_cfg.embed_dim , out_features=d_cfg.vocab_size)

    def forward(self,x):
        return self.unembed(x)

**Looking at the Unembedding layer**

In [364]:
model_unembed = Unembed(m_cfg , d_cfg)
model_unembed.to(device)

model_unembed.eval()
with torch.inference_mode():
    unembed_output = model_unembed(transformer_output)

print(unembed_output.shape)

torch.Size([32, 64, 50255])


* ### Putting together a model to train

In [375]:
class MiniatureGpt(nn.Module):
    def __init__(self , m_cfg:ModelConfig , d_cfg:DataConfig ):
        super().__init__()
        self.embed = Embedding(m_cfg , d_cfg)
        self.transformer_stack = nn.ModuleList([TransformerBlock(m_cfg) for _ in range(m_cfg.N_layers)])
        self.norm = Normalize(m_cfg)
        self.unembed = Unembed(m_cfg , d_cfg)

    def forward(self ,x):
        x = self.embed(x)
        for block in self.transformer_stack:
            x = block(x)
        x = self.norm(x)
        x = self.unembed(x)
        return x #<- these outputs are logits that must be passt to a softmax to get probability of tokens
    
    

**Looking at the MiniatureGpt model**

In [429]:
minigpt_model = MiniatureGpt(m_cfg , d_cfg)
minigpt_model.to(device)

minigpt_model.eval()
with torch.inference_mode():
    minigpt_output = minigpt_model(X)

# print( '----Listing the model state dict:\n' , list( minigpt_model.state_dict() ) )
print(f"----Output shape: {minigpt_output.shape}")
print(f"----An exampel of the output logits:\n {minigpt_output[0,0,:]}")

----Output shape: torch.Size([32, 64, 50255])
----An exampel of the output logits:
 tensor([ 0.5443, -1.1743, -0.5724,  ...,  0.6919,  0.2178,  0.3555],
       device='mps:0')


* # The training

* ### Loss function

In [430]:
loss_fn = nn.CrossEntropyLoss()

* ### Optimizer

In [431]:
optimizer = torch.optim.AdamW(
    minigpt_model.parameters(),
    lr=1e-4,
    betas=(0.9, 0.95),
    weight_decay=0.01
)

* ### Putting the data into X_train ,y_train and X_test , y_test format
  This function selects a batch random points in the training sequence extracts a sequence of tokens with size = context_size and the corresponding labels as the next token. This function will be called for every batch in the training loop.

In [432]:
def get_batch(tokens, batch_size, context_size):

    # Pick random starting positions
    ix = torch.randint(
        len(tokens) - context_size - 1,
        (batch_size,)
    )

    X = torch.stack(
        [tokens[i:i+context_size] for i in ix]
    )

    y = torch.stack(
        [tokens[i+1:i+context_size+1] for i in ix]
    )

    return X, y
    

In [433]:
X_train , y_train = get_batch(train_set, d_cfg.batch_size, d_cfg.context_size)
X_train , y_train = X_train.to(device) , y_train.to(device)
print(X_train.shape)
print(y_train.shape)

with torch.inference_mode():
    logits = minigpt_model(X_train)
b, p, v = logits.shape
loss = loss_fn(
    logits.view(b*p, v),
    y_train.view(b*p)
)
print(loss)


torch.Size([32, 64])
torch.Size([32, 64])
tensor(11.0011, device='mps:0')


In [434]:
epochs = 1

for epoch in range(epochs):
    for batch in range(d_cfg.N_batches):
        X_train , y_train = get_batch(train_set, d_cfg.batch_size, d_cfg.context_size)
        X_train , y_train = X_train.to(device) , y_train.to(device)
        minigpt_model.train()

        logits = minigpt_model(X_train)
        b, p, v = logits.shape
        loss = loss_fn(
            logits.view(b*p, v),
            y_train.view(b*p)
        )
  
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            print(
                f"Epoch {epoch+1}/{epochs}, "
                f"Batch {batch}/{d_cfg.N_batches}, "
                f"Loss: {loss.item():.4f}"
            )
   

Epoch 1/1, Batch 0/226533, Loss: 11.0184
Epoch 1/1, Batch 100/226533, Loss: 5.6107
Epoch 1/1, Batch 200/226533, Loss: 4.6866
Epoch 1/1, Batch 300/226533, Loss: 4.1693
Epoch 1/1, Batch 400/226533, Loss: 3.7533
Epoch 1/1, Batch 500/226533, Loss: 4.1380
Epoch 1/1, Batch 600/226533, Loss: 3.9470
Epoch 1/1, Batch 700/226533, Loss: 3.7623
Epoch 1/1, Batch 800/226533, Loss: 3.6407
Epoch 1/1, Batch 900/226533, Loss: 3.6351
Epoch 1/1, Batch 1000/226533, Loss: 3.6581
Epoch 1/1, Batch 1100/226533, Loss: 3.3250
Epoch 1/1, Batch 1200/226533, Loss: 3.3891
Epoch 1/1, Batch 1300/226533, Loss: 3.5811
Epoch 1/1, Batch 1400/226533, Loss: 3.4784
Epoch 1/1, Batch 1500/226533, Loss: 3.1935
Epoch 1/1, Batch 1600/226533, Loss: 3.2611
Epoch 1/1, Batch 1700/226533, Loss: 3.3460
Epoch 1/1, Batch 1800/226533, Loss: 3.3784
Epoch 1/1, Batch 1900/226533, Loss: 3.1378
Epoch 1/1, Batch 2000/226533, Loss: 3.2187
Epoch 1/1, Batch 2100/226533, Loss: 3.2051
Epoch 1/1, Batch 2200/226533, Loss: 3.1725
Epoch 1/1, Batch 2300/

* # Let's see how the model does in generating text
**I do not expect a lot. It is a miniature gpt.**

In [447]:
# The prompt
prompt = test_set[0:d_cfg.context_size]
tokens = tokenizer.decode(prompt)
print(tokens)

Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled and replied, "Thank you, Spot. I polish it every day."

After playing with the car, Kitty and Spot felt thirsty. They found a small pond with clear water. They


**This function generates text:** 
1. Gets the gpt logits output of the last position devide by temperature (High temperature leads to more stochasticity)
2. Pass the new logits to a softmax function to get probabilities
3. Sample a multinomial distribution to get the new token's index
4. Appends the new token index to the end of the token tensor
5. Do auto regression by passing the sequence of tokens including the newly added one

In [476]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=100, context_size=64, temperature=1.0):
    """
    This function generates new text by running the minigpt model
    
    prompt: list[int] or tensor of token ids
    returns: tensor containing prompt + generated tokens
    """

    model.eval()

    if not torch.is_tensor(prompt):
        tokens = torch.tensor(prompt, dtype=torch.long).unsqueeze(0)
    else:
        tokens = prompt.unsqueeze(0) if prompt.ndim == 1 else prompt

    device = next(model.parameters()).device
    tokens = tokens.to(device)

    for _ in range(max_new_tokens):
     
        # logits: [1, T, vocab_size]
        with torch.inference_mode():
            logits = model(tokens[:,-context_size:])

        # Only use the logits for the last position
        logits = logits[:, -1, :] / temperature

        # Convert to probabilities
        probs = F.softmax(logits, dim=-1)

        # Sample one token
        next_token = torch.multinomial(probs, num_samples=1)

        # Append it
        tokens = torch.cat([tokens, next_token], dim=1)

    return tokens.squeeze(0)

In [486]:
max_new_tokens =45
generated_ids = generate(
    minigpt_model,
    tokenizer,
    prompt,
    max_new_tokens=max_new_tokens,
    context_size = d_cfg.context_size,
    temperature=0.8,
)

print('Prompt text: \n', tokenizer.decode(generated_ids[0:d_cfg.context_size]),'\n')
print('----------------------------------------------------------------------------------------')
print('Generated text: \n', tokenizer.decode(generated_ids[d_cfg.context_size:]),'\n')
print('----------------------------------------------------------------------------------------')
print('True text: \n', tokenizer.decode(test_set[d_cfg.context_size:d_cfg.context_size+max_new_tokens]),'\n')

Prompt text: 
 Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled and replied, "Thank you, Spot. I polish it every day."

After playing with the car, Kitty and Spot felt thirsty. They found a small pond with clear water. They 

----------------------------------------------------------------------------------------
Generated text: 
  drank some water and drank it. The water was cool and cool as they drank. They went to the pond to drink some water. They played in the water all day long.

Kitty and Spot played in the 

----------------------------------------------------------------------------------------
True text: 
  drank the water and felt very happy. They played together all day and became best friends.Once upon a time, in a big forest, there lived a rhinoceros named Roxy. Roxy loved to climb. 



**Go on and run the above cell several times. Due to the probabilistic sampling of the GPT model, we get a different sequence of generated text each time.**